# Gauss elimination, LUP factorization

## Upper triangular matrix system (backward substitution)

```
u11x1 + u12x2 + u13x3 = b1 => x1 = (b1 - u12x2 - u13x3) / u11
0 + u22x2 + u23x3 = b2 => x2 = (b2 - u23x3) / u22
0 + 0 + u33x3 = b3 => x3 = b3 / u33

function x = back_sub(U, b)
    n = length(b);
    x = zeros(n, 1);
    for i = n:-1:1
        x(i) = (b(i) - U(i, i+1:n) * x(i+1:n)) / U(i, i);
    end
end
```



In [ ]:
import numpy as np


def back_sub(U: np.ndarray, b: np.ndarray) -> np.ndarray:
    n = len(b)
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (b[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x

U = np.array([[2, 4, 2], [0, -1, 1], [0, 0, -1]])
b = np.array([8, 0, -1])

x = back_sub(U, b)
print(x)



[1. 1. 1.]


## Forward substitution
```
l11x1 + 0 + 0 = b1 => x1 = b1 / l11
l21x1 + l22x2 + 0 = b2 => x2 = (b2 - l21x1) / l22
l31x1 + l32x2 + l33x3 = b3 => x3 = (b3 - l31x1 - l32x2) / l33

function x = forward_sub(L, b)
    n = length(b);
    x = zeros(n, 1);
    for i = 1:n
        x(i) = (b(i) - L(i, 1:i-1) * x(1:i-1)) / L(i, i);
    end
end
```



In [3]:
import numpy as np


def forward_sub(L: np.ndarray, b: np.ndarray) -> np.ndarray:
    n = len(b)
    x = np.zeros(n)
    for i in range(n):
        x[i] = (b[i] - np.dot(L[i, :i], x[:i])) / L[i, i]
    return x


L = np.array([[1, 0, 0], [2, 1, 0], [3, 2, 1]])
b = np.array([1, 2, 3])

x = forward_sub(L, b)
print(x)



[1. 0. 0.]


## Gaussian elimination


In [7]:
def gaussian_elimination(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    n = len(b)

    for i in range(n - 1):
        # Partial pivoting: place the largest absolute pivot on the diagonal.
        pivot_row = i + np.argmax(np.abs(A[i:, i]))
        if A[pivot_row, i] == 0:
            raise ValueError("Matrix is singular: zero pivot encountered.")

        if pivot_row != i:
            A[[i, pivot_row]] = A[[pivot_row, i]]
            b[[i, pivot_row]] = b[[pivot_row, i]]

        # Eliminate entries below the pivot.
        for j in range(i + 1, n):
            factor = A[j, i] / A[i, i]
            A[j, i:] -= factor * A[i, i:]
            b[j] -= factor * b[i]

    return A, b

A = np.array([[1, -1, 1], [-2, 2, 1], [-3, -1, -5]], dtype=float)
b = np.array([1, 2, -5], dtype=float)

A, b = gaussian_elimination(A, b)
A_ext = np.column_stack((A, b))
print(A_ext)

[[-3.         -1.         -5.         -5.        ]
 [ 0.          2.66666667  4.33333333  5.33333333]
 [ 0.          0.          1.5         2.        ]]


In [10]:
def solve_system_with_gaussian_elimination(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    A, b = gaussian_elimination(A, b)
    print("Matrix after the elimination:")
    print(A)
    print("Right-hand side vector after the elimination:")
    print(b)
    return back_sub(A, b)

A = np.array([[2, 1, -1, -2], [4, 4, 1, 3], [-6, -1, 10, 10], [-2, 1, 8, 4]], dtype=float)
b = np.array([2, 4, -5, 1], dtype=float)
x = solve_system_with_gaussian_elimination(A, b)
print("Solution:")
print(x)


Matrix after the elimination:
[[-6.         -1.         10.         10.        ]
 [ 0.          3.33333333  7.66666667  9.66666667]
 [ 0.          0.          1.6        -3.2       ]
 [ 0.          0.          0.          1.        ]]
Right-hand side vector after the elimination:
[-5.          0.66666667  2.4        -1.        ]
Solution:
[-2.375  4.25  -0.5   -1.   ]


## LUP factorization


Given a matrix A
We compute L, U and P such that P @ A = L @ U
Where L is lower triangular and U is upper triangular.
And P is a permutation matrix.

In [15]:
from scipy.linalg import lu


def lup_factorization(A: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # SciPy returns P_scipy, L, U such that A = P_scipy @ L @ U.
    P_scipy, L, U = lu(A.astype(float), permute_l=False)
    # Convert to the convention used in this notebook: P @ A = L @ U.
    P = P_scipy.T
    return L, U, P


A = np.array([[2, 1, -1, -2], [4, 4, 1, 3], [-6, -1, 10, 10], [-2, 1, 8, 4]], dtype=float)
L, U, P = lup_factorization(A)
print("L:")
print(L)
print("U:")
print(U)
print("P:")
print(P)
print("Check P @ A == L @ U:")
print(np.allclose(P @ A, L @ U))


L:
[[ 1.          0.          0.          0.        ]
 [-0.66666667  1.          0.          0.        ]
 [ 0.33333333  0.4         1.          0.        ]
 [-0.33333333  0.2         0.5         1.        ]]
U:
[[-6.         -1.         10.         10.        ]
 [ 0.          3.33333333  7.66666667  9.66666667]
 [ 0.          0.          1.6        -3.2       ]
 [ 0.          0.          0.          1.        ]]
P:
[[0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]]
Check P @ A == L @ U:
True


In [16]:
from scipy.linalg import lu_factor, lu_solve


def solve_system_with_lup_factorization(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    lu_and_piv = lu_factor(A.astype(float))
    x = lu_solve(lu_and_piv, b.astype(float))
    return x


A = np.array([[2, 1, -1, -2], [4, 4, 1, 3], [-6, -1, 10, 10], [-2, 1, 8, 4]], dtype=float)
b = np.array([2, 4, -5, 1], dtype=float)
x = solve_system_with_lup_factorization(A, b)
print("Solution:")
print(x)


Solution:
[-2.375  4.25  -0.5   -1.   ]
